In [138]:
import pandas as pd

### Phase 1:
prediction  Objective Specification


1. prediction horizon will be on weekly basis (5 days)
2. Given all information available at the close of day t, predict what happens over the next five trading days.
3. SELL = 0
   BUY = 1
   HOLD = -1
4. model can only see data avalilable upto t days
5. For every sample at time t, the model may only use information available at or before the close of day t. It must not access any information from t+1 through t+5 during feature generation or prediction.
6. The model outputs exactly one class (BUY, SELL, or HOLD) for each observation at time t, representing the expected market behavior over the next five trading days.

In [139]:
import os
import sys


In [140]:
project_root = os.path.abspath("..")
sys.path.insert(0,project_root)
from Data.data_loader import load_daily_data

data = pd.read_pickle('../Data/feature_added_data/volume_featured_data.pkl')

In [141]:
import numpy as np

data['previous_close'] = data['Close'].shift(1)
high_low_diff = data['High']-data['Low']
high_prevclose_diff = abs(data['High']-data['previous_close'])
low_prevclose_diff = abs(data['Low']-data['previous_close'])
true_range = pd.concat([high_prevclose_diff,high_low_diff,low_prevclose_diff],axis=1).max(axis=1)
first_atr_value = true_range.iloc[:14].mean()
wilder_atr =[np.nan]*len(data)
atr_value = first_atr_value
wilder_atr[13]=(atr_value)
for i in range(14,len(true_range)):
    wild_atr = ((atr_value*13)+true_range.iloc[i])/14
    wilder_atr[i] =(wild_atr)
    atr_value = wild_atr

data['wilder_atr'] = wilder_atr
data.drop(columns=['previous_close'],inplace=True)


In [142]:
profit_tartet = 2*data['wilder_atr']
stop_loss = 1*data['wilder_atr']
entry_price = data['Close']

upper_barrier = entry_price+profit_tartet
lower_barrier = entry_price - stop_loss
verticle_barrier = [np.nan]*len(data)
for i in range(0,len(data)-5):
 verticle_barrier[i]= data.index[i+5] 
data['upper_barrier'] = upper_barrier
data['lower_barrier'] = lower_barrier
data['verticle_barrier'] = verticle_barrier
data.tail(10)

Price,Close,High,Low,Open,Volume,sma-5,sma-20,sma-50,Close-sma-50,sma_5-sma_20,...,Rolling_minimum_20,Rolling_std,rolling_volatility,RVOL,Volume_MA_20,Volume_Change,wilder_atr,upper_barrier,lower_barrier,verticle_barrier
Date,,,,,,,,,,,,,,,,,,,,,
2026-06-25,227.009995,232.320007,225.550003,232.020004,77674100,234.513998,246.091000,256.4450,-29.435006,-11.577002,...,227.009995,12.234178,0.022150,1.527535,50849315.0,7391500.0,7.955998,242.921991,219.053996,2026-07-02
2026-06-26,232.690002,233.899994,226.130005,227.210007,248372100,232.173999,244.025500,256.1288,-23.438798,-11.851501,...,227.009995,10.660246,0.023135,4.055955,61236400.0,170698000.0,7.942712,248.575426,224.747291,2026-07-06
2026-06-29,240.139999,249.710007,233.800003,234.220001,77619200,233.644000,242.500500,255.9376,-15.797601,-8.856499,...,227.009995,8.643306,0.024756,1.244299,62379880.0,-170752900.0,8.591090,257.322179,231.548910,2026-07-07
2026-06-30,238.339996,241.539993,237.169998,237.350006,66228300,234.489999,241.354499,255.6932,-17.353204,-6.864500,...,227.009995,7.464133,0.023809,1.051058,63011075.0,-11390900.0,8.289583,254.919162,230.050413,2026-07-08
2026-07-01,241.699997,244.899994,234.899994,239.750000,53829900,235.975998,240.613499,255.5616,-13.861603,-4.637502,...,227.009995,6.560239,0.023914,0.846222,63612020.0,-12398400.0,8.411756,258.523508,233.288241,2026-07-09
2026-07-02,242.669998,246.720001,241.080002,241.610001,48246400,239.107999,240.245999,255.4168,-12.746802,-1.138000,...,227.009995,6.201628,0.023345,0.760329,63454630.0,-5583500.0,8.213773,259.097544,234.456225,NaT
2026-07-06,244.160004,246.039993,240.880005,243.800003,40044000,241.401999,239.764500,255.1928,-11.032797,1.637499,...,227.009995,5.419198,0.023101,0.628868,63676340.0,-8202400.0,7.995646,260.151295,236.164358,NaT
2026-07-07,245.979996,248.929993,242.699997,246.979996,40579600,242.569998,239.762000,255.0108,-9.030804,2.807999,...,227.009995,5.416166,0.022142,0.644889,62924930.0,535600.0,7.869528,261.719051,238.110468,NaT
2026-07-08,243.619995,244.800003,240.520004,244.270004,29653000,243.625998,239.681999,254.6034,-10.983405,3.943999,...,227.009995,5.342622,0.022239,0.472840,62712605.0,-10926600.0,7.697418,259.014831,235.922577,NaT


In [143]:
data =data.reset_index()

In [144]:
label = [np.nan]*len(data)
data['Label'] = label

for curr_day in range(0, len(data)-5):
    barrier_hit = False
    entry_price = data['Close'].iloc[curr_day]
    upper_barrier = data['upper_barrier'].iloc[curr_day]
    lower_barrier = data['lower_barrier'].iloc[curr_day]

    for future_day in range(curr_day+1, curr_day+6):
        high = data['High'].iloc[future_day]
        low = data['Low'].iloc[future_day]

        # check stop-loss FIRST -> pessimistic tie-break when both
        # barriers are touched on the same day (can't know true intraday order
        # from daily OHLC, so default to the safer/worse-case assumption)
        if low <= lower_barrier:
            data.loc[curr_day, 'Label'] = 0      # SELL
            barrier_hit = True
            break
        elif high >= upper_barrier:
            data.loc[curr_day, 'Label'] = 1      # BUY
            barrier_hit = True
            break

    if barrier_hit == False:
        data.loc[curr_day, 'Label'] = -1         # HOLD (vertical barrier)

# drop the unlabelable tail rows (last 5 days with no full forward window)
data = data.dropna(subset=['Label']).reset_index(drop=True)

In [145]:
data.tail(10)

Price,Date,Close,High,Low,Open,Volume,sma-5,sma-20,sma-50,Close-sma-50,...,Rolling_std,rolling_volatility,RVOL,Volume_MA_20,Volume_Change,wilder_atr,upper_barrier,lower_barrier,verticle_barrier,Label
2411,2026-06-17,237.500000,245.910004,236.000000,244.899994,44780800,241.916000,254.309002,256.637401,-19.137401,...,12.493330,0.020192,1.064088,42083725.0,9593400.0,7.524448,252.548896,229.975552,2026-06-25,0.0
2412,2026-06-18,244.389999,245.729996,236.020004,240.119995,75624400,242.492001,253.278001,257.100201,-12.710201,...,12.414335,0.020733,1.714128,44118285.0,30843600.0,7.680558,259.751116,236.709441,2026-06-26,0.0
2413,2026-06-22,232.789993,242.000000,232.240005,240.080002,68388500,241.339999,251.494501,257.083000,-24.293007,...,12.677885,0.022483,1.496200,45708125.0,-7235900.0,7.999804,248.789601,224.790189,2026-06-29,1.0
2414,2026-06-23,234.110001,236.869995,232.000000,232.550003,60492600,238.957999,249.884001,256.997600,-22.887600,...,12.741143,0.022653,1.277402,47355980.0,-7895900.0,7.776246,249.662493,226.333755,2026-06-30,0.0
2415,2026-06-24,234.270004,242.419998,232.949997,233.850006,70282600,236.612000,248.333001,256.885201,-22.615196,...,12.654804,0.022701,1.435255,48968705.0,9790000.0,7.897229,250.064461,226.372776,2026-07-01,0.0
2416,2026-06-25,227.009995,232.320007,225.550003,232.020004,77674100,234.513998,246.091000,256.445000,-29.435006,...,12.234178,0.022150,1.527535,50849315.0,7391500.0,7.955998,242.921991,219.053996,2026-07-02,1.0
2417,2026-06-26,232.690002,233.899994,226.130005,227.210007,248372100,232.173999,244.025500,256.128800,-23.438798,...,10.660246,0.023135,4.055955,61236400.0,170698000.0,7.942712,248.575426,224.747291,2026-07-06,1.0
2418,2026-06-29,240.139999,249.710007,233.800003,234.220001,77619200,233.644000,242.500500,255.937600,-15.797601,...,8.643306,0.024756,1.244299,62379880.0,-170752900.0,8.591090,257.322179,231.548910,2026-07-07,-1.0
2419,2026-06-30,238.339996,241.539993,237.169998,237.350006,66228300,234.489999,241.354499,255.693200,-17.353204,...,7.464133,0.023809,1.051058,63011075.0,-11390900.0,8.289583,254.919162,230.050413,2026-07-08,-1.0
2420,2026-07-01,241.699997,244.899994,234.899994,239.750000,53829900,235.975998,240.613499,255.561600,-13.861603,...,6.560239,0.023914,0.846222,63612020.0,-12398400.0,8.411756,258.523508,233.288241,2026-07-09,-1.0


In [146]:
counts = data['Label'].value_counts()

In [147]:
counts[1]

np.int64(486)

In [148]:
len(data)

2421

In [149]:
label_percentage ={}
for i,j in zip(counts,counts.index):
    percentage = (i/len(data))*100
    label_percentage[j] = percentage
    
    

In [150]:
label_percentage

{0.0: 44.81619165634035, -1.0: 35.10945890128046, 1.0: 20.074349442379184}

In [151]:
data.columns.tolist()

['Date',
 'Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'sma-5',
 'sma-20',
 'sma-50',
 'Close-sma-50',
 'sma_5-sma_20',
 'Price > SMA_50',
 'slope_sma-20',
 'EMA_20',
 'EMA_9',
 'EMA_15',
 'EMA_5',
 'EMA_50',
 'EMA_200',
 'RSI_7',
 'RSI_14',
 'RSI_21',
 'macd',
 'macd_signal_line',
 'histogram',
 'SMA_20',
 'STD_20',
 'Upper_Band',
 'Lower_Band',
 'Price_Position',
 'prev_close',
 'ATR',
 'VWAP',
 'OBV',
 'tp',
 'RMF',
 'positive_mf',
 'negative_mf',
 'positive_mf_14',
 'negative_mf_14',
 'MFR',
 'MFI',
 'Daily_return',
 'Weekly_return',
 'Monthly_return',
 'Log_return',
 'Gap_Return',
 'Rolling_mean_20',
 'Rolling_maximum_20',
 'Rolling_minimum_20',
 'Rolling_std',
 'rolling_volatility',
 'RVOL',
 'Volume_MA_20',
 'Volume_Change',
 'wilder_atr',
 'upper_barrier',
 'lower_barrier',
 'verticle_barrier',
 'Label']

In [152]:
data = data.drop(columns=['prev_close',
'tp', 
'RMF',
'positive_mf',
'negative_mf',
'positive_mf_14',
'negative_mf_14',
'MFR',
'Date',
'verticle_barrier'])

In [153]:
data.head()

Price,Close,High,Low,Open,Volume,sma-5,sma-20,sma-50,Close-sma-50,sma_5-sma_20,...,Rolling_minimum_20,Rolling_std,rolling_volatility,RVOL,Volume_MA_20,Volume_Change,wilder_atr,upper_barrier,lower_barrier,Label
0,37.118999,38.941502,35.884998,38.940498,254940000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
1,36.950500,37.162998,36.445000,36.786499,132456000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-122484000.0,NaN,NaN,NaN,-1.0
2,35.953499,37.299999,35.505001,37.275501,146426000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,13970000.0,NaN,NaN,NaN,-1.0
3,37.161999,37.339001,36.299500,36.500000,135116000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-11310000.0,NaN,NaN,NaN,-1.0
4,37.324501,37.493500,36.780499,36.993999,72976000,36.9019,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-62140000.0,NaN,NaN,NaN,-1.0


In [154]:
data.columns.tolist()

['Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'sma-5',
 'sma-20',
 'sma-50',
 'Close-sma-50',
 'sma_5-sma_20',
 'Price > SMA_50',
 'slope_sma-20',
 'EMA_20',
 'EMA_9',
 'EMA_15',
 'EMA_5',
 'EMA_50',
 'EMA_200',
 'RSI_7',
 'RSI_14',
 'RSI_21',
 'macd',
 'macd_signal_line',
 'histogram',
 'SMA_20',
 'STD_20',
 'Upper_Band',
 'Lower_Band',
 'Price_Position',
 'ATR',
 'VWAP',
 'OBV',
 'MFI',
 'Daily_return',
 'Weekly_return',
 'Monthly_return',
 'Log_return',
 'Gap_Return',
 'Rolling_mean_20',
 'Rolling_maximum_20',
 'Rolling_minimum_20',
 'Rolling_std',
 'rolling_volatility',
 'RVOL',
 'Volume_MA_20',
 'Volume_Change',
 'wilder_atr',
 'upper_barrier',
 'lower_barrier',
 'Label']

In [155]:
data.to_pickle('../Data/feature_added_data/three_level_barrier_label.pkl')